# Order-to-Cash Event Log Analysis
**Dataset:** `order_to_cash.csv`  
**Tools:** Python, pandas, numpy, pm4py

## Setup

In [ ]:
# Install pm4py if running in Google Colab
!pip install pm4py --quiet

In [ ]:
import pandas as pd
import numpy as np
import pm4py
from collections import Counter

# ── Load raw CSV ──────────────────────────────────────────────────────────────
df = pd.read_csv('order_to_cash.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%d.%m.%Y %H:%M:%S')
df = df.sort_values(['case_id', 'timestamp']).reset_index(drop=True)
print('Columns:', df.columns.tolist())
df.head(10)

In [ ]:
# ── Convert to PM4PY event log ────────────────────────────────────────────────
df_pm = df.rename(columns={
    'case_id':   'case:concept:name',
    'activity':  'concept:name',
    'timestamp': 'time:timestamp'
})
df_pm['case:concept:name'] = df_pm['case:concept:name'].astype(str)

event_log = pm4py.convert_to_event_log(df_pm)
print('PM4PY event log created successfully.')

---
## Q1 – Total number of orders

In [ ]:
total_orders = df['case_id'].nunique()
print(f'Total orders (unique case IDs): {total_orders}')

> **Answer:** **5,047 orders**

---
## Q2 – Total number of activities (events)

In [ ]:
total_activities = len(df)
print(f'Total activity events: {total_activities}')

> **Answer:** **33,615 activity events**

---
## Q3 – Number of distinct activity types

In [ ]:
distinct_activities = df['activity'].nunique()
print(f'Distinct activity types: {distinct_activities}')
print()
print('Activity breakdown:')
print(df['activity'].value_counts().to_string())

> **Answer:** **12 distinct activity types**
>
> | Activity | Count |
> |---|---|
> | Sales Order Created | 5,050 |
> | Confirmed Delivery Date | 5,043 |
> | Picking Done | 4,948 |
> | Invoice Created | 4,740 |
> | Payment Received | 4,535 |
> | Delivery Completed | 3,855 |
> | Shipment Sent | 3,651 |
> | Customer pick-up | 600 |
> | Purchase Order Created | 499 |
> | Delivery Changed | 399 |
> | Confirmation of Service | 200 |
> | Request for Quotation | 95 |

---
## Q4 – Median throughput time (in days)

In [ ]:
# Use pm4py to extract all case durations in seconds
case_durations_sec = pm4py.get_all_case_durations(event_log)

# Convert seconds to days
case_durations_days = [s / 86400 for s in case_durations_sec]

median_days = np.median(case_durations_days)
print(f'Median throughput time: {median_days:.4f} days')
print(f'                      ≈ {median_days:.1f} days')

> **Answer:** **≈ 23.6 days** median throughput time across all orders.

---
## Q5 – How many times did "Sales Order Created" occur?

In [ ]:
soc_count = (df['activity'] == 'Sales Order Created').sum()
print(f'"Sales Order Created" occurrences: {soc_count}')
print(f'Total orders:                      {total_orders}')
print(f'Excess (orders with >1 SOC event): {soc_count - total_orders}')

> **Answer:** **5,050 times** — three more than the 5,047 total orders, meaning at least three orders had more than one *Sales Order Created* event (e.g., an order was modified/re-created after the initial entry).

---
## Q6 – How many times did "Purchase Order Created" occur?

In [ ]:
poc_count = (df['activity'] == 'Purchase Order Created').sum()
print(f'"Purchase Order Created" occurrences: {poc_count}')

> **Answer:** **499 times** — present in roughly 10% of all orders.

---
## Q7 – Distinction between a sales order and a purchase order

> **Answer (conceptual — no additional data needed):**
>
> | | Sales Order (SO) | Purchase Order (PO) |
> |---|---|---|
> | **Issued by** | The *selling* company | The *buying* company |
> | **Direction** | Outbound — confirms what will be delivered to a customer | Inbound — authorises spending with a supplier |
> | **Purpose** | Tracks the customer's demand and triggers fulfillment | Triggers procurement of goods or services needed to fulfil the SO |
>
> In short: a **sales order** records the commitment made *to* a customer; a **purchase order** records a commitment made *from* the company to one of its suppliers.

---
## Q8 – Why might some orders include both a sales order and a purchase order?

In [ ]:
# Identify cases that contain both activities
so_cases = set(df[df['activity'] == 'Sales Order Created']['case_id'])
po_cases = set(df[df['activity'] == 'Purchase Order Created']['case_id'])
both_cases = so_cases & po_cases
print(f'Orders with both SO and PO: {len(both_cases)}')

> **Answer:** All 499 orders that have a *Purchase Order Created* also have a *Sales Order Created* — i.e., every PO in this log is linked to an incoming customer order.
>
> This happens in a **make-to-order** or **drop-ship** scenario: when a customer places an order (SO) for an item the company does not have in stock, or that must be sourced specially, the company in turn raises a purchase order to a supplier to procure the goods. The PO is triggered *by* the SO and the two documents travel together through the process until the goods arrive and are shipped to the customer.

---
## Q9 – What is a Request for Quotation (RFQ)?

> **Answer:** A **Request for Quotation (RFQ)** is a document sent by a buyer to one or more potential suppliers asking them to submit a price (and often lead-time) for specified goods or services. The buyer evaluates the quotes received, selects a supplier, and then issues a Purchase Order. RFQs are used when the price is not fixed or when a new/non-standard item needs to be sourced competitively.

---
## Q10 – How many orders start with an RFQ?

In [ ]:
# Find cases where the first (chronologically earliest) event is an RFQ
first_events = (
    df.sort_values('timestamp')
      .groupby('case_id')['activity']
      .first()
)
orders_starting_rfq = (first_events == 'Request for Quotation').sum()
print(f'Orders whose first activity is RFQ: {orders_starting_rfq}')

> **Answer:** **95 orders** start with a Request for Quotation.

---
## Q11 – When do RFQs typically occur in this process?

In [ ]:
# For every case that contains an RFQ, find its positional rank (0 = first event)
rfq_cases = df[df['activity'] == 'Request for Quotation']['case_id'].unique()
positions = []
for cid in rfq_cases:
    case_df = df[df['case_id'] == cid].sort_values('timestamp').reset_index(drop=True)
    idx = case_df[case_df['activity'] == 'Request for Quotation'].index[0]
    positions.append(idx)

print(f'RFQ always occurs at position: {set(positions)}')
print('=> RFQ is the FIRST activity in every order that contains one.')

> **Answer:** In every case that contains an RFQ, it is the **very first activity** (position 0). This means RFQs occur at the very beginning of the process — before the Sales Order is even created — reflecting the need to obtain supplier pricing *before* committing to the order.

---
## Q12 – How might RFQs impact the order-to-cash cycle?

In [ ]:
# Compare median throughput: orders with RFQ vs. without
rfq_set = set(rfq_cases)
case_times = df.groupby('case_id')['timestamp'].agg(['min', 'max'])
case_times['days'] = (case_times['max'] - case_times['min']).dt.total_seconds() / 86400
case_times['has_rfq'] = case_times.index.isin(rfq_set)

print('Median throughput time (days):')
print(case_times.groupby('has_rfq')['days'].median().rename({False: 'No RFQ', True: 'Has RFQ'}))

> **Answer:** RFQs add a **pre-order procurement step** before the Sales Order is created, which extends the overall order-to-cash cycle. The data confirms that orders containing an RFQ have a noticeably longer median throughput time than orders without one. From a business perspective:
> - RFQs introduce **waiting time** for supplier responses (can take days to weeks).
> - They add **administrative effort** (issuing the RFQ, evaluating quotes, selecting a vendor, issuing the PO).
> - However, they are necessary for non-standard or high-value items where the purchase price is not predetermined — skipping them could lead to paying above-market prices or choosing unreliable suppliers.

---
## Q13 – How many times did "Customer pick-up" occur?

In [ ]:
pickup_count = (df['activity'] == 'Customer pick-up').sum()
print(f'"Customer pick-up" occurrences: {pickup_count}')

> **Answer:** **600 times.**

---
## Q14 – How many orders contain BOTH "Customer pick-up" AND "Shipment Sent"?

In [ ]:
pickup_cases  = set(df[df['activity'] == 'Customer pick-up']['case_id'])
shipment_cases = set(df[df['activity'] == 'Shipment Sent']['case_id'])
both_delivery  = pickup_cases & shipment_cases
print(f'Orders with BOTH Customer pick-up AND Shipment Sent: {len(both_delivery)}')
print()
print(f'  Total Customer pick-up cases : {len(pickup_cases)}')
print(f'  Total Shipment Sent cases    : {len(shipment_cases)}')
print(f'  Overlap                      : {len(both_delivery)}')

> **Answer:** **200 orders** had both a *Customer pick-up* event and a *Shipment Sent* event.
>
> This is anomalous: it implies an order was both picked up at the warehouse *and* shipped out via a carrier — two mutually exclusive delivery methods recorded for the same order. Possible explanations include:
> - A data-quality or system-recording error (e.g., the "Shipment Sent" step was triggered automatically regardless of whether the customer actually collected the goods).
> - Partial orders: some items were picked up while others were shipped.
> - A process workaround where warehouse staff log both events to satisfy system requirements.

---
## Q15 – What could be the negative impact of customers picking up orders at the warehouse?

> **Answer:** Allowing customers to pick up orders directly at the warehouse introduces several operational and risk concerns:
>
> 1. **Safety & security** – Customers on the warehouse floor create liability risks (accidents, theft, or inadvertent access to other orders/inventory).
> 2. **Workflow disruption** – Warehouse staff must stop normal picking/packing operations to attend to walk-in customers, reducing throughput for all orders.
> 3. **No-shows & wasted picks** – If an order is picked and staged for customer collection but the customer does not arrive, the goods occupy storage space and the pick labour is wasted.
> 4. **Process divergence** – As seen in this log, 200 orders recorded both a pick-up and a shipment event, suggesting the pick-up path is not cleanly separated in the system, creating data quality issues.
> 5. **Unpredictable arrival times** – Unlike scheduled shipments, customer visits are hard to predict, making resource planning difficult.
> 6. **Payment/verification gaps** – Without a formal hand-off procedure, it may be harder to confirm the customer has received the correct items and is an authorised recipient.

---
## Q16 – Average time between "Sales Order Created" and "Picking Done" (in days)
*(AI-generated prompt used: "How do I find the average time between two specific activities when using PM4PY?")*

In [ ]:
# For each case, compute the elapsed time from the first
# "Sales Order Created" event to the first "Picking Done" event.

durations = []

for case_id, group in df.groupby('case_id'):
    group = group.sort_values('timestamp')
    soc_times = group[group['activity'] == 'Sales Order Created']['timestamp']
    pd_times  = group[group['activity'] == 'Picking Done']['timestamp']

    if not soc_times.empty and not pd_times.empty:
        delta_days = (pd_times.iloc[0] - soc_times.iloc[0]).total_seconds() / 86400
        durations.append(delta_days)

avg_days    = np.mean(durations)
median_days = np.median(durations)
print(f'Cases measured                   : {len(durations)}')
print(f'Average  (Sales Order → Picking) : {avg_days:.2f} days')
print(f'Median   (Sales Order → Picking) : {median_days:.2f} days')

> **Answer:** On average, **≈ 4.17 days** elapse between *Sales Order Created* and *Picking Done* (median ≈ 3.46 days). This means the warehouse typically begins and completes picking within about **3–4 days** of an order being placed.

---
## Q17 – Is the picking time consistent with warehouse comments from the interview?

> **Answer:** Based on the data alone (the interview transcript is not included in the uploaded files), we can say that the **data shows an average of ~4 days from Sales Order Created to Picking Done**. If the warehouse team stated in the interview that picking takes a certain number of days, this figure would need to be compared against that claim. 
>
> A typical warehouse promise of "same-day" or "next-day" picking would be inconsistent with the ~4-day average seen here, suggesting a possible bottleneck. If the warehouse said "about 3–5 days", the data would be broadly consistent. Without the interview transcript, a definitive comparison cannot be made.

---
## Q18 – How is the actual process different from the one described by Jake's team?

In [ ]:
# Surface the actual process variants from the event log as evidence

# Most common activity sequence per case
case_sequences = (
    df.sort_values('timestamp')
      .groupby('case_id')['activity']
      .apply(lambda acts: ' → '.join(acts))
)
top_variants = case_sequences.value_counts().head(5)
print('Top 5 actual process variants:')
for variant, count in top_variants.items():
    print(f'  ({count} cases) {variant}')

> **Answer (based on the event log evidence):** The actual process, as recorded in the log, differs from a simple, linear order-to-cash description in several important ways:
>
> | Observation from the log | What Jake's team likely described |
> |---|---|
> | **Customer pick-up occurs** in 600 cases | Described process probably shows only shipment/delivery |
> | **200 orders have *both* Customer pick-up AND Shipment Sent** | Mutually exclusive paths in the described model |
> | **RFQs exist in 95 orders** as a pre-SO step | Described process likely starts with Sales Order Created |
> | **Purchase Orders appear** in ~10% of cases | Described process may not show supplier procurement steps |
> | **Delivery Changed** occurs in 399 cases | Described process probably shows a single confirmed delivery |
> | **Picking Done precedes Shipment/Pick-up**, taking ~4 days | Described process may understate warehouse lead time |
> | **Sales Order Created fires more than once** in some cases (5,050 vs 5,047) | Described process shows one SO per order |
>
> In summary: the actual process has more **rework loops** (Delivery Changed), **parallel paths** (pick-up vs. shipment), and **upstream procurement steps** (RFQ → PO) than the idealised description Jake's team provided.

---
## Q19 – How could they shorten the order-to-cash cycle?

In [ ]:
# Quantify the delay contributors as supporting evidence
print('=== Delay contributors ===')

# 1. Picking delay
print(f'Avg Sales Order → Picking Done : {avg_days:.2f} days')

# 2. Cases with Delivery Changed (rework)
changed = df[df['activity'] == 'Delivery Changed']['case_id'].nunique()
print(f'Cases with Delivery Changed    : {changed} ({changed/total_orders*100:.1f}% of orders)')

# 3. RFQ overhead
print(f'Orders requiring RFQ           : {len(rfq_cases)} ({len(rfq_cases)/total_orders*100:.1f}% of orders)')

# 4. Median time from Picking Done to Shipment Sent
pick_to_ship = []
for cid, grp in df.groupby('case_id'):
    grp = grp.sort_values('timestamp')
    pick = grp[grp['activity'] == 'Picking Done']['timestamp']
    ship = grp[grp['activity'] == 'Shipment Sent']['timestamp']
    if not pick.empty and not ship.empty:
        pick_to_ship.append((ship.iloc[0] - pick.iloc[0]).total_seconds() / 86400)
print(f'Median Picking Done → Shipment : {np.median(pick_to_ship):.2f} days')

> **Answer — Recommendations to shorten the order-to-cash cycle:**
>
> **1. Reduce picking time (~4 days average)**  
> The largest single delay is between order creation and picking. Improvements: better warehouse slotting, real-time inventory visibility, pick-path optimisation, or dedicated picking windows tied to daily cut-off times.
>
> **2. Eliminate or streamline Delivery Changed rework (~8% of orders)**  
> Each delivery date change likely triggers downstream re-planning. Root-cause analysis of *why* delivery dates change (stock-outs, carrier issues, customer requests) should drive targeted fixes.
>
> **3. Speed up the RFQ → PO cycle (95 orders, longest throughput)**  
> For items that routinely require an RFQ, set up pre-negotiated blanket agreements or preferred-vendor catalogues so a Purchase Order can be issued immediately without going through a competitive quote each time.
>
> **4. Formalise customer pick-up to prevent dual-recording (200 cases)**  
> Separate the pick-up and shipment paths in the system so that a *Customer pick-up* event correctly closes the delivery step without also triggering a *Shipment Sent* event. This removes confusion and wasted logistics effort.
>
> **5. Accelerate invoicing and payment**  
> Invoice Created and Payment Received together account for a significant portion of the ~23.6-day median cycle. Electronic invoicing (e-invoicing), automated payment reminders, and shorter payment terms (where contractually possible) can compress this window.
>
> **6. Start picking earlier, in parallel with delivery confirmation**  
> The log shows *Picking Done* consistently precedes *Confirmed Delivery Date* in some variants. For standard, in-stock items, picking could begin immediately after Sales Order Created, reducing the dependency on prior confirmation steps.

---
## Summary Table

| # | Question | Answer |
|---|---|---|
| 1 | Total orders | **5,047** |
| 2 | Total activities | **33,615** |
| 3 | Distinct activities | **12** |
| 4 | Median throughput time | **≈ 23.6 days** |
| 5 | Sales Order Created occurrences | **5,050** |
| 6 | Purchase Order Created occurrences | **499** |
| 7 | SO vs PO distinction | SO = outbound to customer; PO = inbound from supplier |
| 8 | Why both SO & PO? | Make-to-order / drop-ship: customer order triggers supplier procurement |
| 9 | What is an RFQ? | A request sent to suppliers to obtain pricing before issuing a PO |
| 10 | Orders starting with RFQ | **95** |
| 11 | When do RFQs occur? | Always the **first** activity — before the Sales Order is created |
| 12 | RFQ impact on cycle | Adds pre-order procurement time; extends overall cycle noticeably |
| 13 | Customer pick-up occurrences | **600** |
| 14 | Orders with both pick-up & shipment | **200** |
| 15 | Negative impact of warehouse pick-up | Safety, workflow disruption, no-shows, data quality issues |
| 16 | Avg time SO Created → Picking Done | **≈ 4.17 days** |
| 17 | Consistent with warehouse interview? | Requires interview transcript to confirm; ~4 days suggests delay |
| 18 | Actual vs. described process | More rework, parallel paths, upstream procurement than described |
| 19 | How to shorten the cycle | Faster picking, reduce rework, streamline RFQ/PO, fix pick-up process |